# Launch records with SQLite

These ten SQLite queries summarize the supplied launch table. The engine follows the IBM SQLite lab variant. Mission outcomes and first-stage landing outcomes describe different events.

Run the cells in order from the repository root or `notebooks/`. Install `requirements.txt` and run `download_inputs.py` if `data/` is missing. Local output files are written under `results/`.

## Sources and credit

Adapted from this repository's `scripts/03_sql_analysis.py` and the public IBM DS0321EN Skills Network SpaceX labs. IBM and the credited lab authors, including Joseph Santarcangelo, Nayef Abou Tayoun and Yan Luo, retain their source credit. Input URLs and SHA-256 hashes are listed in `SOURCES.json`; see `ATTRIBUTION.md` for third-party terms. This notebook does not grant a new licence over third-party material.


In [1]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
import pandas as pd
pd.set_option('display.max_rows', 12)
pd.set_option('display.max_columns', 12)
pd.set_option('display.max_colwidth', 70)

In [2]:
from pathlib import Path

import os

import matplotlib

matplotlib.use('module://matplotlib_inline.backend_inline')

from IPython.display import display

HERE = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data').is_dir() and (p / 'SOURCES.json').is_file()), None)

if HERE is None:
    raise FileNotFoundError('Open this notebook from the repository root or its notebooks folder; run download_inputs.py first.')

DATA = HERE / 'data'

OUT = HERE / 'results' / '03_sql_sqlite_official_variant'

OUT.mkdir(parents=True, exist_ok=True)

os.chdir(OUT)

import csv, sqlite3

con = sqlite3.connect('spacex_local.sqlite')

cur = con.cursor()

print('SQLite version', sqlite3.sqlite_version)

import pandas as pd

df = pd.read_csv(DATA / 'Spacex.csv')

df.to_sql('SPACEXTBL', con, if_exists='replace', index=False, method='multi')

con.execute('DROP TABLE IF EXISTS SPACEXTABLE')

SQLite version 3.50.4


In [3]:
con.commit()

con.execute('CREATE TABLE SPACEXTABLE AS SELECT * FROM SPACEXTBL WHERE Date IS NOT NULL')

con.commit()

query_1 = 'SELECT DISTINCT Launch_Site FROM SPACEXTABLE ORDER BY Launch_Site'

result_1 = pd.read_sql_query(query_1, con)

display(result_1)

result_1.to_csv(OUT / 'task_1.csv', index=False)

query_2 = "SELECT * FROM SPACEXTABLE WHERE Launch_Site LIKE 'CCA%' LIMIT 5"

result_2 = pd.read_sql_query(query_2, con)

display(result_2)

result_2.to_csv(OUT / 'task_2.csv', index=False)

query_3 = "SELECT SUM(PAYLOAD_MASS__KG_) AS nasa_crs_payload_kg FROM SPACEXTABLE WHERE Customer = 'NASA (CRS)'"

result_3 = pd.read_sql_query(query_3, con)

display(result_3)

result_3.to_csv(OUT / 'task_3.csv', index=False)

query_4 = "SELECT AVG(PAYLOAD_MASS__KG_) AS mean_payload_kg FROM SPACEXTABLE WHERE Booster_Version = 'F9 v1.1'"

result_4 = pd.read_sql_query(query_4, con)

display(result_4)

result_4.to_csv(OUT / 'task_4.csv', index=False)

query_5 = "SELECT MIN(Date) AS first_successful_ground_pad_date FROM SPACEXTABLE WHERE Landing_Outcome = 'Success (ground pad)'"

,Launch_Site
0,CCAFS LC-40
1,CCAFS SLC-40
2,KSC LC-39A
3,VAFB SLC-4E


,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of Brouere cheese",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


,nasa_crs_payload_kg
0,45596


,mean_payload_kg
0,2928.4


In [4]:
result_5 = pd.read_sql_query(query_5, con)

display(result_5)

result_5.to_csv(OUT / 'task_5.csv', index=False)

query_6 = "SELECT Booster_Version FROM SPACEXTABLE WHERE Landing_Outcome = 'Success (drone ship)' AND PAYLOAD_MASS__KG_ > 4000 AND PAYLOAD_MASS__KG_ < 6000"

result_6 = pd.read_sql_query(query_6, con)

display(result_6)

result_6.to_csv(OUT / 'task_6.csv', index=False)

query_7 = 'SELECT Mission_Outcome, COUNT(*) AS launches FROM SPACEXTABLE GROUP BY Mission_Outcome ORDER BY launches DESC'

result_7 = pd.read_sql_query(query_7, con)

display(result_7)

result_7.to_csv(OUT / 'task_7.csv', index=False)

query_8 = 'SELECT Booster_Version, PAYLOAD_MASS__KG_ FROM SPACEXTABLE WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTABLE)'

result_8 = pd.read_sql_query(query_8, con)

display(result_8)

result_8.to_csv(OUT / 'task_8.csv', index=False)

query_9 = "SELECT substr(Date,6,2) AS month, Landing_Outcome,Booster_Version,Launch_Site FROM SPACEXTABLE WHERE substr(Date,1,4)='2015' AND Landing_Outcome = 'Failure (drone ship)'"

result_9 = pd.read_sql_query(query_9, con)

display(result_9)

result_9.to_csv(OUT / 'task_9.csv', index=False)

query_10 = "SELECT Landing_Outcome,COUNT(*) AS outcomes FROM SPACEXTABLE WHERE Date BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY Landing_Outcome ORDER BY outcomes DESC"

,first_successful_ground_pad_date
0,2015-12-22


,Booster_Version
0,F9 FT B1022
1,F9 FT B1026
2,F9 FT B1021.2
3,F9 FT B1031.2


,Mission_Outcome,launches
0,Success,98
1,Success (payload status unclear),1
2,Success,1
3,Failure (in flight),1


,Booster_Version,PAYLOAD_MASS__KG_
0,F9 B5 B1048.4,15600
1,F9 B5 B1049.4,15600
2,F9 B5 B1051.3,15600
3,F9 B5 B1056.4,15600
4,F9 B5 B1048.5,15600
5,F9 B5 B1051.4,15600
6,F9 B5 B1049.5,15600
7,F9 B5 B1060.2,15600
8,F9 B5 B1058.3,15600
9,F9 B5 B1051.6,15600


,month,Landing_Outcome,Booster_Version,Launch_Site
0,01,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
1,04,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40


In [5]:
result_10 = pd.read_sql_query(query_10, con)

display(result_10)

result_10.to_csv(OUT / 'task_10.csv', index=False)

import json

assert con.execute('SELECT COUNT(*) FROM SPACEXTABLE').fetchone()[0] == len(df) == 101

assert int(result_3.iloc[0, 0]) == int(df.loc[df['Customer'].eq('NASA (CRS)'), 'PAYLOAD_MASS__KG_'].sum())

assert result_5.iloc[0, 0] == df.loc[df['Landing_Outcome'].eq('Success (ground pad)'), 'Date'].min()

assert result_7['launches'].sum() == len(df)

assert result_9.shape[0] == len(df[df['Date'].str.startswith('2015') & df['Landing_Outcome'].eq('Failure (drone ship)')])

summary = {'rows': len(df), 'tasks': 10, 'nasa_crs_payload_kg': int(result_3.iloc[0, 0]), 'first_ground_pad_date': result_5.iloc[0, 0], 'engine': 'sqlite3; official IBM SQLite variant'}

(OUT / 'RESULTS.json').write_text(json.dumps(summary, indent=2))

print(json.dumps(summary, indent=2))

con.close()

,Landing_Outcome,outcomes
0,No attempt,10
1,Success (drone ship),5
2,Failure (drone ship),5
3,Success (ground pad),3
4,Controlled (ocean),3
5,Uncontrolled (ocean),2
6,Failure (parachute),2
7,Precluded (drone ship),1


{
  "rows": 101,
  "tasks": 10,
  "nasa_crs_payload_kg": 45596,
  "first_ground_pad_date": "2015-12-22",
  "engine": "sqlite3; official IBM SQLite variant"
}
